# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
# Step 1 — Load the data

from pathlib import Path
import json

# Search the Vocareum workspace for the required data files
workspace = Path("/voc/work")

job_file = next(workspace.rglob("job_snippets.jsonl"), None)
golden_file = next(workspace.rglob("golden_set.jsonl"), None)

if job_file is None:
    raise FileNotFoundError(
        "Could not find job_snippets.jsonl anywhere under /voc/work"
    )

if golden_file is None:
    raise FileNotFoundError(
        "Could not find golden_set.jsonl anywhere under /voc/work"
    )

# The data directory is the folder containing the two files
DATA_DIR = job_file.parent

print("Data directory:", DATA_DIR)
print("Job snippets file:", job_file)
print("Golden set file:", golden_file)

# Load job snippets
snippets = [
    json.loads(line)
    for line in job_file.read_text().splitlines()
    if line.strip()
]

# Load golden answers
golden = {
    row["id"]: row
    for row in (
        json.loads(line)
        for line in golden_file.read_text().splitlines()
        if line.strip()
    )
}

print(f"Loaded {len(snippets)} snippets, {len(golden)} golden entries.")
print("Sample snippet:", snippets[0])

Data directory: /voc/work/AI-RAG_MP1_PromptLab/MP1_PromptLab/data
Job snippets file: /voc/work/AI-RAG_MP1_PromptLab/MP1_PromptLab/data/job_snippets.jsonl
Golden set file: /voc/work/AI-RAG_MP1_PromptLab/MP1_PromptLab/data/golden_set.jsonl
Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the following information from this job posting:

- company
- role
- years_experience_required

Return ONLY a JSON object with exactly these three fields:

{{
  "company": "...",
  "role": "...",
  "years_experience_required": 0
}}

If no specific years-of-experience requirement is stated, use null.
For a range such as "three to five years", use the minimum value.
For fresh graduates or no prior experience, use 0.

Job posting:
{snippet_text}
"""
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include worked examples."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the company, role, and required years of experience from the job posting.

Here are examples showing the expected interpretation.

Example 1:
Job posting:
"TechNova is hiring a Backend Engineer. Candidates should have 5+ years of experience."

Output:
{{
  "company": "TechNova",
  "role": "Backend Engineer",
  "years_experience_required": 5
}}

Example 2:
Job posting:
"Bright Systems is looking for a Junior Developer. This position is open to fresh graduates with no prior experience."

Output:
{{
  "company": "Bright Systems",
  "role": "Junior Developer",
  "years_experience_required": 0
}}

Example 3:
Job posting:
"DataWorks is seeking a Data Analyst with three to five years of experience."

Output:
{{
  "company": "DataWorks",
  "role": "Data Analyst",
  "years_experience_required": 3
}}

Now extract the information from this job posting.

Return ONLY a JSON object with exactly these three fields:

{{
  "company": "...",
  "role": "...",
  "years_experience_required": 0
}}

If no specific years-of-experience requirement is stated, use null.
For a range, use the minimum number of years.

Job posting:
{snippet_text}
"""
        }
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt and explicit JSON schema."""

    return [
        {
            "role": "system",
            "content": """
You are an expert job-posting information extraction assistant.

Your task is to extract exactly three fields from a job posting.

Output schema:
{
  "company": "string",
  "role": "string",
  "years_experience_required": "integer or null"
}

Rules:
1. Extract the company name exactly as stated.
2. Extract the job role/title exactly as stated.
3. Extract the required years of experience as a number.
4. For values such as "5+ years", return 5.
5. For values such as "around 6 years", return 6.
6. For a range such as "three to five years", return the minimum value, 3.
7. Fresh graduates or no prior experience means 0.
8. If no specific years requirement is stated, return null.
9. Do not invent or infer an experience requirement.
10. Return ONLY valid JSON with exactly the three fields.
"""
        },
        {
            "role": "user",
            "content": f"""
Extract the required information from this job posting.

Job posting:
{snippet_text}
"""
        }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the following information from this job posting:

- company
- role
- years_experience_required

First reason through the job posting step by step to identify the correct company, role, and experience requirement.

Pay particular attention to:
- numbers followed by "+" years
- approximate expressions such as "around 6 years"
- ranges such as "three to five years" — use the minimum
- fresh graduates or no prior experience — use 0
- cases where no specific experience requirement is stated — use null
- do not fabricate a value that is not stated

After reasoning, return the final answer as ONLY a JSON object with exactly these fields:

{{
  "company": "...",
  "role": "...",
  "years_experience_required": 0
}}

Job posting:
{snippet_text}
"""
        }
    ]


STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
def parse_response(text: str) -> dict | None:
    # Try to parse a JSON object from the model response.

    if not text:
        return None

    text = text.strip()

    # Remove markdown code fences if present.
    if text.startswith("```"):
        lines = text.splitlines()

        # Remove opening fence such as ```json
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]

        # Remove closing ```
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    # First try parsing the complete response directly.
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    # If there is extra text before the JSON,
    # locate and parse the first JSON object.
    start = text.find("{")

    if start == -1:
        return None

    try:
        parsed, _ = json.JSONDecoder().raw_decode(text[start:])
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        return None

In [5]:
async def run_one(strategy_name: str, snippet: dict) -> dict:
    # Run one strategy on one snippet.

    messages = STRATEGIES[strategy_name](snippet["snippet"])

    start_time = time.perf_counter()

    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )

    latency_s = time.perf_counter() - start_time

    raw_response = response.choices[0].message.content or ""

    extracted = parse_response(raw_response)

    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens

    cost_usd = (
        prompt_tokens * RATES[MODEL]["in"]
        + completion_tokens * RATES[MODEL]["out"]
    )

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw_response": raw_response,
        "extracted": extracted,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost_usd": cost_usd,
        "latency_s": latency_s,
    }


async def run_all() -> list[dict]:
    # Run all 10 x 4 = 40 calls in parallel using asyncio.gather.

    tasks = []

    for snippet in snippets:
        for strategy_name in STRATEGIES:
            tasks.append(run_one(strategy_name, snippet))

    results = await asyncio.gather(*tasks)

    return results

In [6]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'prompt_tokens': 145,
 'completion_tokens': 30,
 'cost_usd': 3.9750000000000004e-05,
 'latency_s': 1.393588404986076}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [8]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    # Compare the three required fields and return 0, 1, 2, or 3.

    if not isinstance(extracted, dict):
        return 0

    def normalise_string(value):
        if value is None:
            return None

        return " ".join(str(value).strip().lower().split())

    def normalise_experience(value):
        if value is None:
            return None

        # Numeric values are already normalised.
        if isinstance(value, (int, float)):
            return int(value)

        text = normalise_string(value)

        if text in {"", "null", "none", "not specified", "not stated"}:
            return None

        text = text.replace("years of experience", "")
        text = text.replace("years experience", "")
        text = text.replace("years", "")
        text = text.strip()

        match = re.search(r"\b(\d+)\s*(?:to|-)\s*(\d+)\b", text)
        if match:
            return int(match.group(1))

        match = re.search(r"\b(\d+)\b", text)
        if match:
            return int(match.group(1))

        number_words = {
            "zero": 0,
            "one": 1,
            "two": 2,
            "three": 3,
            "four": 4,
            "five": 5,
            "six": 6,
            "seven": 7,
            "eight": 8,
            "nine": 9,
            "ten": 10,
        }

        for word, number in number_words.items():
            if re.search(rf"\b{word}\b", text):
                return number

        return None

    score = 0

    if normalise_string(extracted.get("company")) == normalise_string(gold.get("company")):
        score += 1

    if normalise_string(extracted.get("role")) == normalise_string(gold.get("role")):
        score += 1

    if normalise_experience(
        extracted.get("years_experience_required")
    ) == normalise_experience(
        gold.get("years_experience_required")
    ):
        score += 1

    return score

In [9]:
# Apply scoring to all 40 results.

snippet_lookup = {snippet["id"]: snippet["snippet"] for snippet in snippets}

scored = []

for result in results:
    snippet_id = result["snippet_id"]
    extracted = result["extracted"]
    gold = golden[snippet_id]

    scored_result = result.copy()

    scored_result["accuracy"] = score_accuracy(extracted, gold)
    scored_result["parse_success"] = extracted is not None

    scored.append(scored_result)

print(f"Scored {len(scored)} results.")

Scored 40 results.


In [10]:
async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict
) -> int:
    # Use gpt-4o as a judge. Return an integer from 1 to 4.

    judge_prompt = f"""
You are evaluating an information extraction result from a job posting.

Compare the MODEL EXTRACTION against the GOLD ANSWER.

Job posting:
{snippet_text}

MODEL EXTRACTION:
{json.dumps(extracted, ensure_ascii=False)}

GOLD ANSWER:
{json.dumps(gold, ensure_ascii=False)}

Use this rubric:

4 = all three fields are correct.
3 = two of the three fields are correct and there is no fabricated data.
2 = one of the three fields is correct, or the model fabricated a field.
1 = none are correct or the extraction is unparsable.

For years_experience_required:
- Treat values such as "5+" and 5 as equivalent.
- Treat "around 6" and 6 as equivalent.
- Treat "7+" and 7 as equivalent.
- For a range such as "three to five years", use the minimum value.
- Fresh graduates/no prior experience means 0.
- If no specific experience requirement is stated, the correct value is null.

Return ONLY one integer:
1, 2, 3, or 4.
Do not include any explanation.
"""

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": judge_prompt
            }
        ],
        temperature=0.0,
    )

    judge_text = response.choices[0].message.content or ""

    match = re.search(r"\b([1-4])\b", judge_text)

    if match:
        return int(match.group(1))

    return 1

In [11]:
# Run the LLM judge for all 40 results.

snippet_lookup = {snippet["id"]: snippet["snippet"] for snippet in snippets}

judge_tasks = []

for result in scored:
    snippet_id = result["snippet_id"]

    judge_tasks.append(
        score_llm_judge(
            snippet_lookup[snippet_id],
            result["extracted"],
            golden[snippet_id],
        )
    )

judge_scores = await asyncio.gather(*judge_tasks)

for result, judge_score in zip(scored, judge_scores):
    result["llm_judge_score"] = judge_score

print(f"Scored {len(scored)} results.")

Scored 40 results.


In [12]:
print(f"Number of scored results: {len(scored)}")
print("First result judge score:", scored[0]["llm_judge_score"])

Number of scored results: 40
First result judge score: 4


## Step 5 — Build the comparison table

In [13]:
# Step 5 — Build the comparison table

df = pd.DataFrame(scored)

summary = (
    df.groupby("strategy")
    .agg({
        "accuracy": "mean",
        "parse_success": "mean",
        "llm_judge_score": "mean",
        "cost_usd": "sum",
        "latency_s": "median",
    })
    .round(3)
)

summary.columns = [
    "Accuracy (mean of 3)",
    "Parse rate",
    "Judge score",
    "Total cost ($)",
    "Latency p50 (s)",
]

summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.9,0.001,1.145
few_shot,2.9,1.0,4.0,0.001,1.209
structured,2.7,1.0,3.9,0.001,1.044
zero_shot,2.8,1.0,3.9,0.000,1.103


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```